In [1]:
import scanpy as sc
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os
import pandas as pd

In [2]:
adata_vis = sc.read_h5ad("/Users/jk/Documents/Lab2/visium/python_analysis/local_401/nmf/ann_finest_level/nfact12/h5ad/adata_vis_assigned_cell_types_nfact12_assigned_factors_log_minmax.h5ad")


In [3]:
adata_vis = adata_vis[adata_vis.obs['cond']!="UNC"].copy()


In [4]:
for i in range(4,20):
    mclust = pd.read_csv(f"/Users/jk/Documents/Lab2/visium/python_analysis/lambda/csv/mclust_{i}_no_UNC.csv")  # will just be one column
    adata_vis.obs[f'mclust_{i}'] = mclust.iloc[:, 0].values
    mclust_refined = pd.read_csv(f"/Users/jk/Documents/Lab2/visium/python_analysis/lambda/csv/mclust_refined_{i}_no_UNC.csv")  # will just be one column
    adata_vis.obs[f'mclust_refined_{i}'] = mclust_refined.iloc[:, 0].values

In [5]:
for col in adata_vis.obs.columns:
    if col.startswith("mclust"):
        adata_vis.obs[col] = adata_vis.obs[col].astype("category")

In [6]:
adata_vis.uns["spatial"].keys()

dict_keys(['06_30914_A1', '08_38774_B2', '11_13888_A1', '12_39986_A2', '16_39724_B1', '16_46257_A1', '16_53837_A10', '17_25789_B1', '17_35291_B1', '18_23779_A2', '18_57617_A1', '19_18542_A4', '19_35057_C3', '19_48719_A1', '20_12743_C1', '20_17688_B2', '20_22642_A1', '20_24241_A2', '20_26330_B3', '20_28197_A1', '20_33362_C4', '20_33940_B2', '20_41501_C1', '20_41615_B1', '20_41847_A1', '21_05738_A1', '21_06301_B2', '21_24095_A3', '21_24837_A1', '21_25528_A3', '21_55244_B1', '21_55747_C3', '21_57231_A3', '22_16220_B1', '22_18440_A2', '22_18446_A1', '22_50637_A1', '23_15209_A3', '23_41922_B2', '23_45450_A3', '23_50343_B2', '24_10794_B1', '24_23755_A1', '24_27523_C5'])

In [7]:
adata_vis[adata_vis.obs['cond']!="UNC"].obs['library_id'].unique()

['18_57617_A1', '20_33940_B2', '20_24241_A2', '19_35057_C3', '20_17688_B2', ..., '08_38774_B2', '16_39724_B1', '21_24095_A3', '21_55747_C3', '22_16220_B1']
Length: 28
Categories (28, object): ['06_30914_A1', '08_38774_B2', '11_13888_A1', '16_39724_B1', ..., '23_15209_A3', '23_41922_B2', '23_50343_B2', '24_27523_C5']

In [8]:
library_ids = adata_vis[adata_vis.obs['cond']!="UNC"].obs['library_id'].unique().tolist()
library_ids

['18_57617_A1',
 '20_33940_B2',
 '20_24241_A2',
 '19_35057_C3',
 '20_17688_B2',
 '20_28197_A1',
 '20_22642_A1',
 '20_41501_C1',
 '20_33362_C4',
 '20_41615_B1',
 '21_06301_B2',
 '21_57231_A3',
 '22_18440_A2',
 '23_15209_A3',
 '23_50343_B2',
 '23_41922_B2',
 '20_26330_B3',
 '21_24837_A1',
 '06_30914_A1',
 '19_18542_A4',
 '24_27523_C5',
 '11_13888_A1',
 '17_25789_B1',
 '08_38774_B2',
 '16_39724_B1',
 '21_24095_A3',
 '21_55747_C3',
 '22_16220_B1']

In [9]:
Image.MAX_IMAGE_PIXELS = None  # disables the decompression bomb check

In [ ]:
import os
from PIL import Image
import pandas as pd

# Base save folder
#save_dir = "patches_multiscale"
#os.makedirs(save_dir, exist_ok=True)

# List of libraries (exclude "UNC")
library_ids = adata_vis[adata_vis.obs['cond'] != "UNC"].obs['library_id'].unique().tolist()

# Define pixel-based scales (since 0.5um = 1 px)
# Example: Visium spot = 110 pixels (55 µm × 2 for context)
# spot_radius_px = 55  # 55 µm = 110 pixels for 1× patch diameter
scales = {
    "55um": 55,
    "100um": 100,
#    "145um": 145,
#    "200um": 200,
#     "250um": 250,
#     "300um": 300,
#    "350um": 350,
#    "400um": 400,
#    "500um": 500
}


# Optional: save metadata for traceability
metadata_dict = {scale_name: [] for scale_name in scales.keys()}

for lib_id in library_ids:
    image_path = f"/Users/jk/Documents/Lab2/visium/spaceranger_input/tiff/{lib_id.replace('_','-')}.tif"
    image = Image.open(image_path).convert("RGB")

    coords = adata_vis.obsm["spatial"][adata_vis.obs['library_id'] == lib_id]  # (n_spots, 2)
    cluster_labels = adata_vis[adata_vis.obs['library_id'] == lib_id].obs["mclust_refined_6"].tolist()

    for i, (x, y) in enumerate(coords):
        label = cluster_labels[i]

        for scale_name, radius_px in scales.items():
            # Crop bounding box
            left = int(x - radius_px )
            top = int(y - radius_px)
            right = int(x + radius_px)
            bottom = int(y + radius_px)

            patch = image.crop((left, top, right, bottom))


            # Create folder by cluster + scale
            cluster_dir = os.path.join(f"visium_patches/{scale_name}", f"cluster{label}")
            os.makedirs(cluster_dir, exist_ok=True)

            # Construct patch filename and path
            patch_filename = f"{lib_id}_spot_{i}_{scale_name}.png"
            patch_path = os.path.join(cluster_dir, patch_filename)

            # Save resized patch
            patch.save(patch_path)

            # Store metadata
            metadata_dict[scale_name].append({
                "patch_path": patch_path,
                "library_id": lib_id,
                "spot_index": i,
                "cluster": label,
                "scale": scale_name,
                "x_coord": x,
                "y_coord": y,
                "left": left,
                "top": top,
                "right": right,
                "bottom": bottom
            })

# ------------------------------------
# Save one CSV per scale
# ------------------------------------
for scale_name, metadata in metadata_dict.items():
    metadata_df = pd.DataFrame(metadata)
    csv_path = f"extract_patches_metadata_{scale_name}.csv"
    metadata_df.to_csv(csv_path, index=False)
    print(f"✅ Saved metadata for {scale_name} to {csv_path}")


